In [1]:
#putanja do glavnog direktorija projekta
from pathlib import Path

if Path.cwd().name == "notebooks":
    %cd ..

c:\Users\Korisnik\Desktop\ZNANSTVENO_PROGRAMIRANJE\projekt\2025-sci-prog\projects\synteticnewswithLLMs-ldanolic


In [2]:
#biblioteke
import pandas as pd

In [3]:
#uvoz funkcija za pripremu podataka iz preprocessing.py
from src.preprocessing import (
    load_isot_data,
    split_dataset,
    save_splits
)

In [4]:
#putanje do izvornih podataka
RAW_DIR = Path("data/raw")

FAKE_PATH = RAW_DIR / "Fake.csv"
TRUE_PATH = RAW_DIR / "True.csv"

print("Fake.csv postoji:", FAKE_PATH.exists())
print("True.csv postoji:", TRUE_PATH.exists())

Fake.csv postoji: True
True.csv postoji: True


In [ ]:
#ucitavanje, ciscenje i spajanje originalnih fake i true podataka (iz preprocessing.py) 
data = load_isot_data(
    FAKE_PATH,
    TRUE_PATH
) 

print("Ukupan broj clanaka:", len(data))
data.head()

Ukupan broj clanaka: 39086


,title,text,full_text,label,original_type,source_id
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON ( ) - The head of a conservative Re...,"As U.S. budget fight looms, Republicans flip t...",0,real,isot_00000
1,U.S. military to accept transgender recruits o...,WASHINGTON ( ) - Transgender people will be al...,U.S. military to accept transgender recruits o...,0,real,isot_00001
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON ( ) - The special counsel investiga...,Senior U.S. Republican senator: 'Let Mr. Muell...,0,real,isot_00002
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON ( ) - Trump campaign adviser George...,FBI Russia probe helped by Australian diplomat...,0,real,isot_00003
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON ( ) - President Donald Trum...,Trump wants Postal Service to charge 'much mor...,0,real,isot_00004


In [6]:
#osnovni pregled tih podataka (iz Fake.csv i True.csv)
print("Stupci:")
print(data.columns.tolist())

print("\nBroj redaka i stupaca:")
print(data.shape)

print("\nNedostajuce vrijednosti:")
print(data.isnull().sum())

Stupci:
['title', 'text', 'full_text', 'label', 'original_type', 'source_id']

Broj redaka i stupaca:
(39086, 6)

Nedostajuce vrijednosti:
title            0
text             0
full_text        0
label            0
original_type    0
source_id        0
dtype: int64


In [7]:
print("Raspodjela labela:") #koliko ima clanka koji su fake, a koliko koji su true

print(
    data["label"]
    .value_counts()
    .sort_index()
)

print("\nPostotci:")

print(
    data["label"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

Raspodjela labela:
label
0    21194
1    17892
Name: count, dtype: int64

Postotci:
label
0    54.224019
1    45.775981
Name: proportion, dtype: float64


In [8]:
#podjela podataka na train i test skup 
SEED = 42

train_df, test_df = split_dataset(
    data=data,
    test_size=0.20,
    random_state=SEED
)

print("TRAIN:", train_df.shape)
print("TEST:", test_df.shape)

TRAIN: (31268, 6)
TEST: (7818, 6)


In [9]:
#provjera raspodjele klasa u svakom skupu, zbog koristenja stratify
def show_distribution(name, df):

    print("\n", name)
    print(
        df["label"].value_counts(normalize=True).sort_index()* 100
    )

show_distribution("TRAIN",train_df)
show_distribution("TEST",test_df)


 TRAIN
label
0    54.224767
1    45.775233
Name: proportion, dtype: float64

 TEST
label
0    54.221028
1    45.778972
Name: proportion, dtype: float64


In [10]:
#pregled broja real i fake vrijednosti u train i test skupu
summary = pd.DataFrame({
    "dataset": [
        "Train",
        "Test"
    ],

    "total": [
        len(train_df), len(test_df)
    ],

    "real": [
        (train_df["label"] == 0).sum(),
        (test_df["label"] == 0).sum()
    ],

    "fake": [
        (train_df["label"] == 1).sum(),
        (test_df["label"] == 1).sum()
    ]
})

summary

,dataset,total,real,fake
0,Train,31268,16955,14313
1,Test,7818,4239,3579


In [11]:
#putanja za spremanje tih obradenih podataka
PROCESSED_DIR = Path("data/processed")

In [12]:
#spremanje train i test skupova
save_splits(
    train_df,
    test_df,
    PROCESSED_DIR
)